In [ ]:
#   numpy(넘파이)는 숫자 여러 개를 '배열(array)'로 묶어 한 번에 계산하게 해 주는 라이브러리입니다.
#   이 노트북에서는 데이터 준비/정규화 같은 'NumPy 흐름'에 사용합니다.
import numpy as np

#   torch는 PyTorch 라이브러리입니다.
#   이번 노트북에서는 다음을 쓰기 위해 사용합니다.
#       (1) torch.nn.Module           : PyTorch 모델을 만들기 위한 기본 class
#       (2) torch.nn.Linear           : H(x)=a·X_norm + b 계산을 대신해 주는 부품(model 안에 넣음)
#       (3) torch.nn.BCEWithLogitsLoss: Binary Cross Entropy Cost 계산을 대신해 주는 부품
#       (4) 자동 미분(autograd)과 optimizer(torch.optim.SGD)
#
#   참고: 이 노트북에서는 'import torch.nn as nn' 같은 줄임 import를 쓰지 않습니다.
#       항상 torch.nn.Module, torch.nn.Linear, torch.nn.BCEWithLogitLoss처럼 전체 이름을 적습니다.
import torch

In [2]:
#   ========================================
#   1. 입력값 X와 정답 y 준비   (이전 파일과 동일)
#   ========================================

#   X: 입력값입니다. 여기서는 사람의 키(cm)를 사용합니다.
X = np.array([160, 170, 180, 190])

#   y: 정답값입니다. 0은 농구선수 아님, 1은 농구선수입니다.
#       X와 y는 순서대로 짝지어집니다. 즉 키 160 → 정답 0, 키 190 → 정답 1.
y = np.array([0, 0, 1, 1])

print('입력값 X: ', X)
print('입력값 y: ', y)

입력값 X:  [160 170 180 190]
입력값 y:  [0 0 1 1]


In [3]:
#   ========================================
#   2. 입력값 정규화    (이전 파일과 동일)
#   ========================================

#   평균(mean)과 표준편차(std)를 계산합니다.
X_mean = np.mean(X)
X_std = np.std(X)

#   정규화 공식: (원본값 - 평균) / 표준편차
#   주의: 실제 학습에는 원래 키 X가 아니라, 정규화된 입력값 X_norm을 사용합니다.
#         (X_mean, X_std는 나중에 '새 입력값 예측'에서도 똑같이 다시 씁니다.)
X_norm = (X - X_mean) / X_std

print('입력값 평균 X_mean: ', X_mean)
print('입력값 표준편차 X_std: ', X_std)
print('정규화된 입력값 X_norm: ', X_norm)

입력값 평균 X_mean:  175.0
입력값 표준편차 X_std:  11.180339887498949
정규화된 입력값 X_norm:  [-1.34164079 -0.4472136   0.4472136   1.34164079]


In [4]:
#   ========================================
#   2-1. X_norm 과 y를 PyTorch tensor 로 변환하고 shape을 (n, 1)로 정리
#   ========================================

#   dtype = torch.float32: 소수점 계산(미분)을 위해 실수(float) 형식으로 만듭니다.
X_norm_tensor = torch.tensor(X_norm, dtype = torch.float32)
y_tensor = torch.tensor(y, dtype = torch.float32)

#   torch.nn.Linear(1, 1)에 넣으려면 각 데이터가 '입력 특성 1개'를 가진 형태,
#   즉 shape (n, 1) 이어야 합니다. 그래서 reshape(-1, 1)로 모양을 바꿉니다.
#       -1: 행 개수는 알아서 (여기서는 4)
#        1: 열 개수는 1 (입력 특성 1개)
X_norm_tensor = X_norm_tensor.reshape([-1, 1])
y_tensor = y_tensor.reshape([-1, 1])

print('학습용 입력 tensor X_norm_tensor:\n', X_norm_tensor)
print('학습용 정답 tensor y_tensor:\n', y_tensor)

#   shape을 꼭 확인합니다. 둘 다 (4, 1) 이어야 합니다.
print('X_norm_tensor shape: ', X_norm_tensor.shape)
print('y_tensor shape: ', y_tensor.shape)

학습용 입력 tensor X_norm_tensor:
 tensor([[-1.3416],
        [-0.4472],
        [ 0.4472],
        [ 1.3416]])
학습용 정답 tensor y_tensor:
 tensor([[0.],
        [0.],
        [1.],
        [1.]])
X_norm_tensor shape:  torch.Size([4, 1])
y_tensor shape:  torch.Size([4, 1])


In [5]:
#   ========================================
#   3. PerceptronModel 정의 (forward가 H를 반환)
#   ========================================

#   torch.nn.Module을 상속받아 우리만의 모델 class를 만듭니다.  (구조는 이전 파일과 동일)
class PerceptronModel(torch.nn.Module):

    def __init__(self):
        #   super().__init__(): 부모 class인 torch.nn.Module의 초기화를 먼저 실행합니다.
        #                       이 줄이 있어야 PyTorch가 self.linear의 weight, bias를
        #                       '학습 대상 파라미터'로 제대로 등록하고 관리합니다.  (반드시 필요)
        super().__init__()
        
        #   self.linear       : H(x) = a·X_norm + b를 계산하는 선형 부품입니다.
        #                       기존 강의의 a는 self.linear.weight, b는 self.linear.bias 입니다.
        self.linear = torch.nn.Linear(1, 1)
        
    def forward(self, x):
        #   forward        : 입력 x가 들어왔을 때 실제 계산 순서를 정의하는 함수입니다.
        
        #   H(x) = a·X_norm + b (sigmoid 전의 선형 계산값 - 확률이 아닙니다.)
        H = self.linear(x)
        
        #   이번 파일은 torch.nn.BCEWithLogitsLoss()를 쓰므로,
        #   sigmoid를 적용하지 않고 H를 그대로 반환합니다.  (z가 아니라 H!)
        return H

In [6]:
#   ========================================
#   4. model 생성
#   ========================================

#   torch.manual_seed(42)
#       - model 생성 시 내부의 self.linear.weight(=a), self.linear.bias(=b)가 랜덤 초기화되기 때문입니다.
torch.manual_seed(42)

#   model = PerceptronModel(): 우리가 정의한 class로 실제 모델 객체를 만듭니다.
model = PerceptronModel()

#   model 안에 자동으로 만들어진 weight(=a)와 bias(=b)의 '학습 전' 초기값을 확인합니다.
#   model.linear.weight 는 기존 강의의 a 역할, model.linear.bias는 기존 강의의 b 역할입니다.
print('model.linear.weight: ', model.linear.weight)
print('model.linear.bias: ', model.linear.bias)
print('model 구조: ', model)

model.linear.weight:  Parameter containing:
tensor([[0.7645]], requires_grad=True)
model.linear.bias:  Parameter containing:
tensor([0.8300], requires_grad=True)
model 구조:  PerceptronModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)


In [7]:
#   ========================================
#   5. 학습 전 model 출력 확인 (구조 점검용)
#   ========================================

#   이전 파일의 model은 H를 반환합니다. 그래서 예측 확률을 보려면 직접 sigmoid를 적용합니다.
#   아직 학습 전이라 weight, bias는 랜덤 초기값 상태입니다. (예측이 맞을 필욘느 없습니다.)
with torch.no_grad():
    H_test = model(X_norm_tensor)
    z_test = torch.sigmoid(H_test)
    
    
#   H와 z(확률)는 둘 다 각 데이터에 대해 하나씩 나오므로 (4, 1) shape이어야 합니다.
print('H_test shape: ', H_test.shape)
print('H_test: \n', H_test)
print('z_test shape: ', z_test.shape)
print('z_test: \n', z_test)

H_test shape:  torch.Size([4, 1])
H_test: 
 tensor([[-0.1957],
        [ 0.4881],
        [ 1.1719],
        [ 1.8557]])
z_test shape:  torch.Size([4, 1])
z_test: 
 tensor([[0.4512],
        [0.6197],
        [0.7635],
        [0.8648]])


In [8]:
#   ========================================
#   6. 학습 설정, criterion(BCEWithLogitsLoss), optimizer 생성
#   ========================================

#   learning_rate(학습률): 한 번에 weight, bias를 얼마나 크게 수정할지 정하는 값입니다.
learning_rate = 0.1

#   epochs(에폭): 전체 데이터를 몇 번 반복해서 학습할지 정하는 값입니다.
epochs = 1000

#   torch.nn.BCEWithLogitsLoss()
#       - sigmoid와 BCE Cost 계산을 내부에서 함께 처리하는 부품입니다.
#       - 사용법: Cost = criterion(H, y_tensor) ← z가 아니라 H를 넣습니다!
criterion = torch.nn.BCEWithLogitsLoss()

#   torch.optim.SGD(model.parameters(), lr= learning_rate)
#       - model.parameters(): model 안의 self.linear.weight(=a)와 self.linear.bias(=b)를 가져옵니다.
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

print('learning_rate: ', learning_rate)
print('epoches: ', epochs)
print('criterion: ', criterion)
#   이 출력 결과에 보이는 값들이 optimizer가 업데이트할 학습 대상입니다.
print(list(model.parameters()))

learning_rate:  0.1
epoches:  1000
criterion:  BCEWithLogitsLoss()
[Parameter containing:
tensor([[0.7645]], requires_grad=True), Parameter containing:
tensor([0.8300], requires_grad=True)]


In [9]:
#   ========================================
#   7. BCEWithLogitsLoss 로 경사 하강법 학습    (이번 실습의 핵심 루프)
#   ========================================

for epoch in range(epochs):
    #   ----- 1. 이전 epoch에서 계산된 grad 초기화 -----
    #   optimizer는 model.parameters()(= weight, bias)를 관리하므로,
    #   이 한 줄이 model 내부 두 파라미터의 grad를 한꺼번에 0으로 만듭니다.
    optimizer.zero_grad()
    
    #   ----- 2. model 호출하면 내부적으로 forward 실행 -----
    #   이번 파일의 forward는 sigmoid를 적용하지 않고 H를 반환합니다.
    #   즉 H = self.linear(X_norm_tensor) 결과가 그대로 H 입니다. (선현 계산값, 확률 아님)
    H = model(X_norm_tensor)
    
    #   ----- 3. BCEWithLogitsLoss로 Cost 계산 -----
    #   주의: 여기서 z = torch.sigmoid(H)를 직접 만들지 않습니다.   (그건 BCELoss 방식)
    mean_cost = criterion(H, y_tensor)
    
    #   ----- 4. backward: model 내부 파라미터의 grad 자동 계산 -----
    #       model.linear.weight.grad    (= 기존 grad_a)
    #       model.linear.bias.grad      (= 기존 grad_b)
    #   에 저장합니다.
    mean_cost.backward()
    
    #   ----- 5. optimizer가 model 내부 weight와 bias 업데이트 -----
    optimizer.step()
    
    #   ----- 6. 학습 상태 출력 -----
    #   입력 특성이 1개라 weight, bias에 값이 하나씩만 있으므로 .item()으로 숫자만 꺼냅니다.
    if epoch % 100 == 0 or epoch == epochs - 1:
        print(
            f'epoch = {epoch}, '
            f'Cost = {mean_cost.item():.6f}, '
            f'weight(a) = {model.linear.weight.item():.6f}, '
            f'bias(b) = {model.linear.bias.item():.6f}, '
        )
        
    #   (참고) 초반 3 epoch 에서만 grad 값을 확인해 봅니다.
    #       기존 grad_a → model.linear.weight.grad
    #       기존 grad_b → model.linear.bias.grad
    if epoch < 3:
        print(
            f'  (확인용) model.linear.weight.grad = {model.linear.weight.grad.item():.6f}'
            f'model.linear.bias.grad = {model.linear.bias.grad.item():.6f}'
        )

epoch = 0, Cost = 0.495464, weight(a) = 0.793780, bias(b) = 0.812529, 
  (확인용) model.linear.weight.grad = -0.292415model.linear.bias.grad = 0.174793
  (확인용) model.linear.weight.grad = -0.286153model.linear.bias.grad = 0.169918
  (확인용) model.linear.weight.grad = -0.280072model.linear.bias.grad = 0.165171
epoch = 100, Cost = 0.178670, weight(a) = 2.290082, bias(b) = 0.173212, 
epoch = 200, Cost = 0.125357, weight(a) = 3.002210, bias(b) = 0.061586, 
epoch = 300, Cost = 0.099283, weight(a) = 3.509002, bias(b) = 0.026837, 
epoch = 400, Cost = 0.082901, weight(a) = 3.912263, bias(b) = 0.013229, 
epoch = 500, Cost = 0.071398, weight(a) = 4.250606, bias(b) = 0.007116, 
epoch = 600, Cost = 0.062789, weight(a) = 4.543496, bias(b) = 0.004091, 
epoch = 700, Cost = 0.056068, weight(a) = 4.802371, bias(b) = 0.002480, 
epoch = 800, Cost = 0.050660, weight(a) = 5.034644, bias(b) = 0.001570, 
epoch = 900, Cost = 0.046207, weight(a) = 5.245449, bias(b) = 0.001031, 
epoch = 999, Cost = 0.042507, weight(a

In [10]:
#   ========================================
#   8. 학습 완료 후 최종 weight(a), bias(b) 확인
#   ========================================

#   입력 특성이 1개라 값이 하나씩만 있으므로 .item()으로 숫자만 꺼냅니다.
print('학습된 weight(a): ', model.linear.weight.item())
print('학습된 bias(b): ', model.linear.bias.item())

# tensor 원본 형태도 함께 확인해 둡니다.    (shape과 requires_grad 표시를 볼 수 있습니다.)
print('model.linear.weight: ', model.linear.weight)
print('model.linear.bias: ', model.linear.bias)

#   학습 후 grad도 한 번 확인해 봅니다. (마지막 epoch의 grad가 남아 있습니다.)
#       기존 grad_a → model.linear.weight.grad
#       기존 grad_b → model.linear.bias.grad
print('model.linear.weight.grad: ', model.linear.weight.grad)
print('model.linear.bias.grad: ', model.linear.bias.grad)

학습된 weight(a):  5.436657428741455
학습된 bias(b):  0.0007013113354332745
model.linear.weight:  Parameter containing:
tensor([[5.4367]], requires_grad=True)
model.linear.bias:  Parameter containing:
tensor([0.0007], requires_grad=True)
model.linear.weight.grad:  tensor([[-0.0185]])
model.linear.bias.grad:  tensor([2.6390e-05])


In [11]:
#   ========================================
#   9. 새로운 입력값 예측
#   ========================================

#   키가 185cm인 사람이 농구선수인지 예측합니다.
input_height = 185

#   새로운 입력값도 학습 데이터와 '같은 기준'으로 정규화해야 합니다.
#   학습 때 사용한 X_mean, X_std를 그대로 다시 사용합니다.  (새로 계산하면 안 됩니다.)
input_norm = (input_height - X_mean) / X_std

#   예측은 학습이 아니므로 weight, bias를 업데이트하지 않습니다.
#   따라서 미분 계산 기록도 필요 없으므로 with torch.no_grad() 안에서 계산합니다.
with torch.no_grad():
    #   torch.nn.Linear(1, 1)에 넣으려면 입력 shape을 (1, 1)로 맞춰야 합니다.
    input_norm_tensor = torch.tensor([[input_norm]], dtype = torch.float32)
    print('input_norm_tensor shape', input_norm_tensor.shape)
    
    #   model은 H를 반환합니다. (선형 계산값 - 확률 아님)
    H_new = model(input_norm_tensor)
    
    #   예측 확률을 보려면 H에 sigmoid를 직접 적용합니다.
    #   probablity = sigmoid(H_new) 이며, 이것이 기존 수식의 z와 같은 의미입니다.
    probability = torch.sigmoid(H_new)
    
    #   0.5 이상이면 1(농구선수), 미만이면 0(농구선수 아님)
    pred = 1 if probability.item() >= 0.5 else 0
    
#   H_new           : 선형 계산값
#   probability     : sigmoid(H_new)로 계산한 예측 확률 (=기존 수식의 z)
#   pred            : probability가 0.5 이상인지에 따라 정한 최종 분류 결과
print('H_new: ', H_new.item())
print('probability: ', probability.item())
print('prediction: ', pred)

print(f'\n키가 {input_height}cm인 사람의 농구선수 확률(probabilityh): {probability.item():.4f}')

input_norm_tensor shape torch.Size([1, 1])
H_new:  4.863395690917969
probability:  0.9923350214958191
prediction:  1

키가 185cm인 사람의 농구선수 확률(probabilityh): 0.9923


### 자습용 체크리스트 - 개념 체크 1/2
#### 개념 체크리스트 (H, z, ligits)입니다:
- H(x)는 무엇인가?
    이진 분류를 위해 우선 변수들을 바탕으로 작성된 선형 수식입니다.
- z는 무엇인가?
    H(x)에 대해 sigmoid를 적용하여 변환한 확률값입니다.
- H(x)는 확률인가?
    선형 회귀식입니다.
- z는 확률인가?
    네, H(x)에 대하여 sigmoid 과정을 통해 산출된 확률값입니다.

### 자습용 체크리스트 - 개념 체크 2/2
#### 개념 체크리스트 (Cost, mean_cost, sigmoid 위치):
- logits는 이번 교재에서 무엇으로 이해하면 되는가?
    sigmoid를 적용하기 전 선형 회귀식 H(x)입니다.
- Cost와 mean_cost는 어떤 관계인가?
    Cost에 대해 각 데이터의 평균을 반영한 것이 mean_cost 입니다.
- sigmoid는 완전히 사라진 것인가, 위치가 바뀐 것인가?
    표면상으로는 사라졌지만, BCEWithLogitsLoss()에 내재화되어 반영됩니다.
- mean_cost는 무엇을 의미하는가?
    예측확률과 실제 값의 차이를 의미합니다.

### 자습용 체크리스트 - 코드 흐름 체크 1/2
#### 코드 흐름 체크리스트(BCELoss, BCEWithLogitsLoss, forward):
- BCEWithLogitsLoss에는 H를 넣는가? z를 넣는가?
    선형 회귀식 H를 넣습니다.
- 이번 강의에서 forward()는 무엇을 반환하는가?
    선형 회귀식 H를 반환합니다.
- forward()에서 sigmoid를 제거한 이유는 무엇인가?
    BCEWithLogitsLoss 자체적으로 회귀식 H(x)에 대해 sigmoid를 취해줍니다.
- mean_cost.backward()는 무엇을 계산하는가?
    weight(a) 및 bias(b)에 대한 미분값을 계산해줍니다.

### 자습용 체크리스트 - 코드 흐름 체크 2/2
#### 코드 흐름 체크리스트(backward, optimizer.step, 예측단계):
- optimizer.step()은 무엇을 업데이트하는가?
    weight(a) 및 bias(b)를 업데이트 합니다.
- 학습 단계와 예측 단계의 차이는 무엇인가?
    학습과 달리 예측은 새로운 입력값에 대하여 별도의 학습 없이 기존의 분석 모델을 통해 분석 결과를 도출합니다.
- 이번 강의에서 forward()는 무엇을 반환하는가?
    H(x), 선형 회귀식을 반환합니다.
- forward()에서 sigmoid를 제거한 이유는 무엇인가?
    기존 따론 계산을 진행했던 sigmoid()가 BCEWithLogitsLoss() 내에 내면화되어 진행되기 때문에 제거했습니다.
- mean_cost.backward()는 무엇을 계산하는가?
    weight(a) 및 bias(b)의 미분값을 계산합니다.